# SEL选股策略工厂类 WtSelStraFact
```cpp
class WtSelStraFact : public ISelStrategyFact
```
该类实现了 ISelStrategyFact 接口，是SEL策略的工厂类。负责创建、管理和删除SEL策略实例，支持策略的动态加载和插件化开发。

## 获取工厂名称 getName
```cpp
/**
 * @brief 获取工厂名称的实现
 * @return const char* 返回策略工厂的名称字符串
 * 
 * 该函数返回策略工厂的名称，用于标识和管理不同的策略工厂。
 * 返回的名称是常量字符串"WtSelStraFact"，在系统中应该是唯一的。
 * 
 * @note 返回的是常量字符串指针，不需要调用者释放内存
 */
const char* WtSelStraFact::getName()
{
	return FACT_NAME;
}
```

## 创建策略实例 createStrategy
```cpp
/**
 * @brief 创建策略实例的实现
 * @param name 策略名称，用于指定要创建的策略类型
 * @param id 策略唯一标识符，用于在系统中唯一标识该策略实例
 * @return SelStrategy* 返回创建的策略对象指针，如果策略名称不存在则返回NULL
 * 
 * 该函数根据策略名称创建对应的策略对象实例。
 * 当前支持的策略类型：
 * - "DualThrustSelection": 创建DualThrust选股策略实例
 * 
 * 如果传入的策略名称不匹配任何已知策略，则返回NULL。
 * 
 * @note 调用者负责管理返回的指针，使用完毕后应通过deleteStrategy删除
 */
SelStrategy* WtSelStraFact::createStrategy(const char* name, const char* id)
{
	if (strcmp(name, "DualThrustSelection") == 0) // 比较策略名称是否为"DualThrustSelection"
		return new WtStraDtSel(id); // 创建DualThrust选股策略实例并返回
	return NULL;
}
```

## 枚举策略名称 enumStrategy
```cpp
/**
 * @brief 枚举策略名称的实现
 * @param cb 枚举策略名称的回调函数，每枚举到一个策略都会调用此回调
 * 
 * 该函数枚举工厂中所有可用的策略类型，通过回调函数通知调用者。
 * 当前工厂支持的策略：
 * - "DualThrustSelection": DualThrust选股策略
 * 
 * 回调函数会被调用一次，传入以下参数：
 * - factName: 工厂名称（"WtSelStraFact"）
 * - straName: 策略名称（"DualThrustSelection"）
 * - isLast: 是否为最后一个策略（true，因为只有一个策略）
 * 
 * @note 如果将来添加更多策略，需要在此函数中添加更多的回调调用
 */
void WtSelStraFact::enumStrategy(FuncEnumSelStrategyCallback cb)
{
	cb(FACT_NAME, "DualThrustSelection", true);
}
```

## 删除策略实例 deleteStrategy
```cpp
/**
 * @brief 删除策略实例的实现
 * @param stra 要删除的策略对象指针
 * @return bool 删除成功返回true，失败返回false
 * 
 * 该函数删除指定的策略对象，释放相关资源。
 * 删除前会进行以下检查：
 * 1. 检查策略指针是否为空，如果为空则直接返回true（视为成功）
 * 2. 检查策略是否属于本工厂创建，通过比较策略的工厂名称
 * 3. 只有属于本工厂的策略才会被删除，其他策略返回false
 * 
 * @note 删除后策略指针将失效，调用者不应再使用该指针
 */
bool WtSelStraFact::deleteStrategy(SelStrategy* stra)  // 删除策略函数实现
{
	if (stra == NULL)
		return true;

	if (strcmp(stra->getFactName(), FACT_NAME) != 0) // 检查策略是否属于本工厂创建
		return false; // 如果不属于本工厂，返回false（拒绝删除）

	delete stra; // 删除策略对象，调用析构函数释放资源
	return true;
}
```

# DualThrust选股策略类 WtStraDtSel
```cpp
class WtStraDtSel : public SelStrategy
```
DualThrust 选股策略是基于 DualThrust 双突破算法的多标的选股策略。

## 成员
- **策略指标参数**
  - `double _k1`：上轨系数，用于计算上突破轨道，通常取值范围为0.5-1.5
  - `double _k2`：下轨系数，用于计算下突破轨道，通常取值范围为0.5-1.5
  - `uint32_t _days`：回看天数，用于计算价格波动范围，通常取值为5-20
- **数据周期相关参数**
  - `std::string _period`：K线周期，如"m1"（1分钟）、"m5"（5分钟）、"d1"（日线）等
  - `uint32_t _count`： K线条数，用于获取历史K线数据，通常取值为_days的2-3倍
  - `bool _isstk`：是否为股票标志，true表示股票模式（不支持做空），false表示期货模式（支持做空）
- **合约代码相关参数**
  - `std::unordered_set<std::string> _codes`：合约代码集合，存储策略要选股的所有标的代码，使用无序集合提高查找效率

## 获取策略名称 getName
```cpp
/**
 * @brief 获取策略名称的实现
 * @return const char* 返回策略的名称字符串
 * 
 * 该函数返回策略的名称，用于标识策略类型。
 * 返回值为"DualThrustSelection"，表示这是DualThrust选股策略。
 */
const char* WtStraDtSel::getName() 
{
	return "DualThrustSelection";
}
```

## 获取所属策略工厂名称 getFactName
```cpp
/**
 * @brief 获取所属策略工厂名称的实现
 * @return const char* 返回策略所属的工厂名称字符串
 * 
 * 该函数返回策略所属的策略工厂名称，用于标识策略的来源工厂。
 * 返回值为"WtSelStraFact"，表示该策略由WtSelStraFact工厂创建。
 * 
 * @note FACT_NAME常量定义在WtSelStraFact.cpp中，值为"WtSelStraFact"
 */
const char* WtStraDtSel::getFactName()
{
	return FACT_NAME;
}
```

## 策略初始化 init
```cpp
/**
 * @brief 策略初始化实现
 * @param cfg 策略配置参数，包含策略运行所需的所有参数
 * @return bool 初始化成功返回true，失败返回false
 * 
 * 该函数从配置参数中加载策略运行所需的参数，包括：
 * - days: 回看天数，用于计算价格波动范围（必需参数）
 * - k1: 上轨系数，用于计算上突破轨道（必需参数）
 * - k2: 下轨系数，用于计算下突破轨道（必需参数）
 * - period: K线周期，如"m1"、"m5"、"d1"等（必需参数）
 * - count: K线条数，用于获取历史K线数据（必需参数）
 * - codes: 合约代码列表，逗号分隔的多个合约代码（必需参数）
 * - stock: 是否为股票，true表示股票模式（不支持做空），false表示期货模式（可选参数，默认为false）
 * 
 * @note 如果配置参数为空，函数返回false
 * @note 如果配置参数中缺少必要参数，可能导致运行时错误
 */
bool WtStraDtSel::init(WTSVariant* cfg)
{
	if (cfg == NULL)
		return false;

	_days = cfg->getUInt32("days");
	_k1 = cfg->getDouble("k1");
	_k2 = cfg->getDouble("k2");
	_period = cfg->getCString("period");
	_count = cfg->getUInt32("count");

	_isstk = cfg->getBoolean("stock");

	std::string codes = cfg->getCString("codes");
	auto ayCodes = StrUtil::split(codes, ",");
	for (auto& code : ayCodes)
		_codes.insert(code);

	return true;
}
```

## 策略初始化完成回调 on_init
```cpp
/**
 * @brief 策略初始化完成回调实现
 * @param ctx SEL策略上下文对象，提供数据访问和交易执行接口
 * 
 * 该函数在策略初始化完成后被调用，用于执行初始化后的准备工作。
 * 主要功能包括：
 * 1. 订阅所有标的的Tick数据，用于接收实时行情
 * 
 * @note 该函数重写了SelStrategy基类的虚函数
 */
void WtStraDtSel::on_init(ISelStraCtx* ctx)
{
	for(auto& code : _codes) // 遍历所有标的代码
	{
		ctx->stra_sub_ticks(code.c_str()); // 订阅标的的Tick数据，确保能接收到实时行情数据
	}
}
```

## 策略调度执行入口 on_schedule
**多标的选股策略的核心调度引擎**。它在特定时间点被调度触发（例如每分钟、每日），负责遍历策略管理的所有标的（Codes），对每一个标的独立执行数据获取、指标计算、信号判断，并最终通过**目标仓位管理**接口执行交易。
* **遍历资产池**
  * 遍历初始化时解析的合约代码集合 `_codes`。
  * 策略对集合中的每一个 `curCode` 独立执行以下逻辑，互不干扰。
* **交易时段过滤**
  * 调用 `ctx->stra_get_sessinfo` 获取该标的的交易时段信息。
  * 检查当前调度时间 `uTime` 是否在交易时段内 (`isInTradingTime`)。
  * **逻辑**：如果当前不在交易时间（如休息时段或非盘中），直接 `continue` 跳过该标的，不进行计算。
* **数据准备**
  * **格式调整**：如果是股票模式 (`_isstk`)，在代码后追加后缀（如 `-`）以适配内部数据格式。
  * **拉取 K 线**：调用 `stra_get_bars` 获取指定周期 (`_period`) 和长度 (`_count`) 的历史 K 线。
  * **异常处理**：如果 K 线指针为空或长度为 0，释放资源并跳过。
* **核心算法计算**
  * **确定参数**：根据 `_days` 确定回溯窗口。
  * **计算历史极值**（基于过去 `_days` 天，排除当前 Bar）：
    * `hh`：历史最高价。
    * `ll`：历史最低价。
    * `hc`：历史收盘价的最大值。
    * `lc`：历史收盘价的最小值。
  * **计算当前轨道**：
    * `Range` (波动区间) = `max(hh - lc, hc - ll)`。
    * 获取当前 Bar 的**开盘价** (`openPx`)。
    * `上轨` = `openPx` + `_k1` * `Range`。
    * `下轨` = `openPx` - `_k2` * `Range`。
* **信号判定与持仓调整**
  * 获取当前 Bar 的**最高价** (`highPx`) 和 **最低价** (`lowPx`) 用于判定是否发生突破。
  * 获取当前该标的的持仓 `curPos`。
  * **空仓状态 (Pos == 0)**：
    * 向上突破 (`highPx >= 上轨`)：调用 `stra_set_position` 设置目标持仓为正（买入）。
    * 向下突破 (`lowPx <= 下轨`) 且非股票：调用 `stra_set_position` 设置目标持仓为负（卖出）。
  * **持多状态 (Pos > 0)**：
    * 向下突破 (`lowPx <= 下轨`)：调用 `stra_set_position` 设置目标持仓为 0（止损/反转平仓）。
  * **持空状态 (Pos < 0)**：
    * 向上突破 (`highPx >= 上轨`) 且非股票：调用 `stra_set_position` 设置目标持仓为 0（止损/反转平仓）。
* **资源清理**
  * **强制释放**：循环结束前必须调用 `kline->release()`，防止内存泄漏。
```cpp
/**
 * @brief 策略调度执行入口实现
 * @param ctx SEL策略上下文对象，提供数据访问和交易执行接口
 * @param uDate 当前日期，格式为YYYYMMDD
 * @param uTime 当前时间，格式为HHMMSS
 * @note 该函数重写了SelStrategy基类的虚函数
 */
void WtStraDtSel::on_schedule(ISelStraCtx* ctx, uint32_t uDate, uint32_t uTime)
```

## Tick数据处理回调 on_tick
```cpp
/**
 * @brief Tick数据处理回调实现
 * @param ctx SEL策略上下文对象，提供数据访问和交易执行接口
 * @param stdCode 标准合约代码，触发Tick数据的合约
 * @param newTick 新的Tick数据，包含最新的价格和成交量信息
 * 
 * 该函数在接收到新的Tick数据时被调用，用于处理实时市场数据。
 * DualThrust选股策略主要基于K线数据进行交易决策，因此Tick数据处理为空实现。
 * 
 * 如果将来需要基于Tick数据做更精细的交易决策（如：
 * - 基于Tick级别的价格变化进行更精确的入场/出场
 * - 基于Tick级别的成交量进行过滤
 * - 基于Tick级别的买卖盘口进行决策），可以在此函数中实现。
 * 
 * @note 该函数重写了SelStrategy基类的虚函数
 */
void WtStraDtSel::on_tick(ISelStraCtx* ctx, const char* stdCode, WTSTickData* newTick) {}
```

## K线闭合回调 on_bar
```cpp
/**
 * @brief K线闭合回调实现
 * @param ctx SEL策略上下文对象，提供数据访问和交易执行接口
 * @param stdCode 标准合约代码，触发K线数据的合约
 * @param period K线周期，如"m1"、"m5"等
 * @param newBar 新的K线数据
 * 
 * 该函数在K线闭合时被调用，用于处理K线数据。
 * DualThrust选股策略主要基于定时调度执行，因此K线数据处理为空实现。
 * 
 * @note 该函数重写了SelStrategy基类的虚函数
 */
void WtStraDtSel::on_bar(ISelStraCtx* ctx, const char* stdCode, const char* period, WTSBarStruct* newBar) {}
```